In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tsfel
import os

# Set style visualisasi plot
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [2]:
# 1. Membaca Dataset Mentah CO (365 Hari)
raw_path = '../data/csv/CO_gresik_timeseries.csv'
df_raw = pd.read_csv(raw_path)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values('date').reset_index(drop=True)

total_rows = len(df_raw)
nan_awal = df_raw['CO'].isna().sum()
valid_awal = df_raw['CO'].notna().sum()
print(f"Total Baris Observasi   : {total_rows} hari")
print(f"Jumlah Data Valid Awal  : {valid_awal} hari ({valid_awal/total_rows*100:.2f}%)")
print(f"Jumlah Missing (NaN)    : {nan_awal} hari ({nan_awal/total_rows*100:.2f}%)")

# 2. Deteksi Outlier Menggunakan Metode Interquartile Range (IQR) pada Data Valid
q1 = df_raw['CO'].quantile(0.25)
q3 = df_raw['CO'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

is_outlier = (df_raw['CO'] < lower_bound) | (df_raw['CO'] > upper_bound)
jumlah_outlier = is_outlier.sum()
print(f"\nKuartil 1 (Q1)           : {q1:.6f} mol/m²")
print(f"Kuartil 3 (Q3)           : {q3:.6f} mol/m²")
print(f"IQR (Q3 - Q1)            : {iqr:.6f} mol/m²")
print(f"Batas Bawah (Lower)      : {lower_bound:.6f} mol/m²")
print(f"Batas Atas (Upper)       : {upper_bound:.6f} mol/m²")
print(f"Outlier Terdeteksi       : {jumlah_outlier} hari")

# Menampilkan rincian tanggal & nilai outlier yang terdeteksi secara dinamis
df_outliers = df_raw[is_outlier][['date', 'CO']].copy()
df_outliers['date'] = df_outliers['date'].dt.strftime('%Y-%m-%d')
print("\nDaftar Tanggal & Nilai Outlier Terdeteksi:")
print(df_outliers.to_string(index=False))

# 3. Mengosongkan Nilai Outlier Menjadi NaN
df_prep = df_raw.copy()
df_prep.loc[is_outlier, 'CO'] = np.nan
nan_setelah_outlier = df_prep['CO'].isna().sum()
print(f"\nJumlah NaN Setelah Outlier Dikosongkan: {nan_setelah_outlier} hari ({nan_setelah_outlier/total_rows*100:.2f}%)")

# 4. Imputasi Linear Time Interpolation Sekaligus
df = df_prep.copy()
df['CO_clean'] = df['CO'].interpolate(method='linear', limit_direction='both')
nan_akhir = df['CO_clean'].isna().sum()
print(f"Jumlah NaN Setelah Imputasi Akhir      : {nan_akhir} hari (100% Clean!)")


Total Baris Observasi   : 365 hari
Jumlah Data Valid Awal  : 192 hari (52.60%)
Jumlah Missing (NaN)    : 173 hari (47.40%)

Kuartil 1 (Q1)           : 0.026530 mol/m²
Kuartil 3 (Q3)           : 0.030774 mol/m²
IQR (Q3 - Q1)            : 0.004244 mol/m²
Batas Bawah (Lower)      : 0.020164 mol/m²
Batas Atas (Upper)       : 0.037140 mol/m²
Outlier Terdeteksi       : 11 hari

Daftar Tanggal & Nilai Outlier Terdeteksi:
      date       CO
2025-09-14 0.017361
2025-09-24 0.041642
2025-10-08 0.039112
2025-10-09 0.045237
2026-04-05 0.017621
2026-05-05 0.017592
2026-05-12 0.016968
2026-06-13 0.018679
2026-07-15 0.019071
2026-08-10 0.038350
2026-08-14 0.039100

Jumlah NaN Setelah Outlier Dikosongkan: 184 hari (50.41%)
Jumlah NaN Setelah Imputasi Akhir      : 0 hari (100% Clean!)


In [3]:
# Simpan Data Super Clean ke File CSV yang Simpel & Mudah Diingat
df_clean = df[['date', 'CO_clean']].rename(columns={'CO_clean': 'CO'})
clean_path = '../data/csv/CO_clean.csv'
df_clean.to_csv(clean_path, index=False)

print(f"SUCCESS: Dataset bersih disimpan ke '{clean_path}' ({len(df_clean)} baris data)")

SUCCESS: Dataset bersih disimpan ke '../data/csv/CO_clean.csv' (365 baris data)


In [4]:
# 1. Memuat Katalog Konfigurasi Fitur TSFEL
cfg = tsfel.get_features_by_domain()

# Penyesuaian max_width=2 pada 4 fitur wavelet agar menghasilkan 1 skala gelombang yang pas (tidak kosong)
wavelet_feats = ['Wavelet absolute mean', 'Wavelet energy', 'Wavelet standard deviation', 'Wavelet variance']
for feat in wavelet_feats:
    if feat in cfg.get('spectral', {}):
        cfg['spectral'][feat]['parameters']['max_width'] = 2

print("Katalog fitur TSFEL berhasil dimuat dan disesuaikan!")

Katalog fitur TSFEL berhasil dimuat dan disesuaikan!


In [5]:
# 2. Mengekstrak Fitur TSFEL dari Sinyal CO Clean (365 Hari)
features_df = tsfel.time_series_features_extractor(cfg, df_clean['CO'], fs=1, verbose=0)

# 3. Inisiasi Daftar 68 Fitur Utama Sesuai Format Tugas Dosen
FEATURE_LIST = [
    'abs_energy', 'auc', 'autocorr', 'average_power', 'calc_centroid', 'calc_max', 'calc_mean',
    'calc_median', 'calc_min', 'calc_std', 'calc_var', 'dfa', 'distance', 'ecdf', 'ecdf_percentile',
    'ecdf_percentile_count', 'ecdf_slope', 'entropy', 'fundamental_frequency', 'higuchi_fractal_dimension',
    'hist_mode', 'human_range_energy', 'hurst_exponent', 'interq_range', 'kurtosis', 'lempel_ziv',
    'lpcc', 'max_frequency', 'max_power_spectrum', 'maximum_fractal_length', 'mean_abs_deviation',
    'mean_abs_diff', 'mean_diff', 'median_abs_deviation', 'median_abs_diff', 'median_diff',
    'median_frequency', 'mfcc', 'mse', 'negative_turning', 'neighbourhood_peaks',
    'petrosian_fractal_dimension', 'pk_pk_distance', 'positive_turning', 'power_bandwidth', 'rms',
    'skewness', 'slope', 'spectral_centroid', 'spectral_decrease', 'spectral_distance',
    'spectral_entropy', 'spectral_kurtosis', 'spectral_positive_turning', 'spectral_roll_off',
    'spectral_roll_on', 'spectral_skewness', 'spectral_slope', 'spectral_spread', 'spectral_variation',
    'spectrogram_mean_coeff', 'sum_abs_diff', 'wavelet_abs_mean', 'wavelet_energy', 'wavelet_entropy',
    'wavelet_std', 'wavelet_var', 'zero_cross'
]

# 4. Format Matriks Horizontal 68 Kolom (Baris 1: Header Nama Fitur, Baris 2: Nilai Angka)
row_dict = {}
for feat in FEATURE_LIST:
    matched = []
    for c in features_df.columns:
        c_clean = c.lower().replace('0_', '').replace(' ', '_').replace('-', '_')
        f_clean = feat.lower().replace(' ', '_').replace('-', '_')
        if c_clean.startswith(f_clean) or f_clean.startswith(c_clean):
            matched.append(c)
    val = features_df[matched].values.mean() if matched else 0.0
    row_dict[feat] = val

df_68_matrix = pd.DataFrame([row_dict])

# 5. Simpan File CSV Matriks 68 Fitur (Format Direct Header & Value)
features_path = '../data/csv/CO_tsfel_features.csv'
df_68_matrix.to_csv(features_path, index=False)
print(f"SUCCESS: Matriks 68 Fitur TSFEL berhasil disimpan ke '{features_path}'! (Shape: {df_68_matrix.shape})")

SUCCESS: Matriks 68 Fitur TSFEL berhasil disimpan ke '../data/csv/CO_tsfel_features.csv'! (Shape: (1, 68))


In [6]:
# 6. Tampilkan Pratinjau Matriks 68 Fitur TSFEL
df_68_matrix

,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,0.0,0.0,3.0,0.000837,0.0,0.0,0.0,0.0,0.0,0.0,...,0.127599,0.73457,0.000012,0.0,0.0,0.00242,2.117515,0.0,0.000006,0.0
